# Load Raw Data

Loads `offers.json`, `profile.json`, and `transactions.json` from `data/raw/` into Spark DataFrames.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ifood-coupons-attribution")
    .master("local[*]")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/07 07:39:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pathlib import Path

RAW_DIR = Path.cwd().parent / "data" / "raw"

offers_path = str(RAW_DIR / "offers.json")
profile_path = str(RAW_DIR / "profile.json")
transactions_path = str(RAW_DIR / "transactions.json")

RAW_DIR, offers_path, profile_path, transactions_path

(PosixPath('/home/caioolubini/Projects/ifood-coupons-atribution/data/raw'),
 '/home/caioolubini/Projects/ifood-coupons-atribution/data/raw/offers.json',
 '/home/caioolubini/Projects/ifood-coupons-atribution/data/raw/profile.json',
 '/home/caioolubini/Projects/ifood-coupons-atribution/data/raw/transactions.json')

In [3]:
# Files are JSON arrays (not JSON Lines), so multiLine=True is required
df_offers = spark.read.option("multiLine", True).json(offers_path)
df_profile = spark.read.option("multiLine", True).json(profile_path)
df_transactions = spark.read.option("multiLine", True).json(transactions_path)

## Offers

In [4]:
df_offers.printSchema()
df_offers.show(10, truncate=False)
print("rows:", df_offers.count())

root
 |-- channels: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- discount_value: long (nullable = true)
 |-- duration: double (nullable = true)
 |-- id: string (nullable = true)
 |-- min_value: long (nullable = true)
 |-- offer_type: string (nullable = true)



+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|channels                    |discount_value|duration|id                              |min_value|offer_type   |
+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|[email, mobile, social]     |10            |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10       |bogo         |
|[web, email, mobile, social]|10            |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10       |bogo         |
|[web, email, mobile]        |0             |4.0     |3f207df678b143eea3cee63160fa8bed|0        |informational|
|[web, email, mobile]        |5             |7.0     |9b98b8c7a33c4b65b9aebfe6a799e6d9|5        |bogo         |
|[web, email]                |5             |10.0    |0b1e1539f2cc45b7b9fa7c272da2e1d7|20       |discount     |
|[web, email, mobile, social]|3             |7.0     |2298d6c36e964ae4a3e7e9706d1fb8c2|7        |discoun

rows: 10


## Profile

In [5]:
df_profile.printSchema()
df_profile.show(10, truncate=False)
print("rows:", df_profile.count())

root
 |-- age: long (nullable = true)
 |-- credit_card_limit: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- id: string (nullable = true)
 |-- registered_on: string (nullable = true)

+---+-----------------+------+--------------------------------+-------------+
|age|credit_card_limit|gender|id                              |registered_on|
+---+-----------------+------+--------------------------------+-------------+
|118|NULL             |NULL  |68be06ca386d4c31939f3a4f0e3dd783|20170212     |
|55 |112000.0         |F     |0610b486422d4921ae7d2bf64640c50b|20170715     |
|118|NULL             |NULL  |38fe809add3b4fcf9315a9694bb96ff5|20180712     |
|75 |100000.0         |F     |78afa995795e4d85b5d9ceeca43f5fef|20170509     |
|118|NULL             |NULL  |a03223e636434f42ac4c3df47e8bac43|20170804     |
|68 |70000.0          |M     |e2127556f4f64592b11af22de27a7932|20180426     |
|118|NULL             |NULL  |8ec6ce2a7e7949b1bf142def7d0e0586|20170925     |
|118|NULL      

rows: 17000


## Transactions

In [6]:
df_transactions.printSchema()
df_transactions.show(10, truncate=False)
print("rows:", df_transactions.count())

root
 |-- account_id: string (nullable = true)
 |-- event: string (nullable = true)
 |-- time_since_test_start: double (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- offer id: string (nullable = true)
 |    |-- offer_id: string (nullable = true)
 |    |-- reward: double (nullable = true)



+--------------------------------+--------------+---------------------+----------------------------------------------------+
|account_id                      |event         |time_since_test_start|value                                               |
+--------------------------------+--------------+---------------------+----------------------------------------------------+
|78afa995795e4d85b5d9ceeca43f5fef|offer received|0.0                  |{NULL, 9b98b8c7a33c4b65b9aebfe6a799e6d9, NULL, NULL}|
|a03223e636434f42ac4c3df47e8bac43|offer received|0.0                  |{NULL, 0b1e1539f2cc45b7b9fa7c272da2e1d7, NULL, NULL}|
|e2127556f4f64592b11af22de27a7932|offer received|0.0                  |{NULL, 2906b810c7d4411798c6938adc9daaa5, NULL, NULL}|
|8ec6ce2a7e7949b1bf142def7d0e0586|offer received|0.0                  |{NULL, fafdcd668e3743c1bb461111dcafc2a4, NULL, NULL}|
|68617ca6246f4fbc85e91a2a49552598|offer received|0.0                  |{NULL, 4d5c57ea9a6940dd891ad53e9dbe8da0, NULL, NULL}|


rows: 306534


In [7]:
df_transactions.groupBy("event").count().show(truncate=False)

+---------------+------+
|event          |count |
+---------------+------+
|transaction    |138953|
|offer received |76277 |
|offer completed|33579 |
|offer viewed   |57725 |
+---------------+------+

